## Macromolecular model

This case study follows from the model presented in:

*Omta et al. 2017. Extracting physiological traits from batch and chemostate culture data. L&O*

The equations are below.

\begin{equation} \frac{dC}{dt} = (C_{syn} - E)P \end{equation}
\begin{equation} \frac{dP}{dt} = P_{syn}P \end{equation}
\begin{equation} \frac{dr}{dt} = \frac{1}{\tau}(r_0 - r) \end{equation}
\begin{equation} \frac{dN}{dt} = -\frac{dP}{dt} \end{equation}

\begin{equation} P_{syn} = \mu\left(\frac{N}{N+K}\right) \end{equation}
\begin{equation} r_0 = b\frac{P}{C} \end{equation}
\begin{equation} r_{cell} = \frac{C}{P} \end{equation}
\begin{equation} E = \frac{1}{2}m_{ex}\left(1+\tanh\left(r_{cell} - r_{ex}\right)\right) \end{equation}

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)
    

### Data Loading

In [ ]:
dat = pd.read_csv('../../data/macromolecular/macromolecules.csv')
dat_flynn = pd.read_csv('../../data/macromolecular/flynn_macromolecules.csv')

dat['N']   *= 0.001
dat['chl'] *= 1000
dat['C']   *= 0.001

dat = dat.merge(dat_flynn.loc[:, ('t', 'ammonium')], on='t')
dat['ammonium'] *= 0.08325909

dat = dat.rename(columns={'N': 'PR', 'ammonium': 'N', 'C': 'CH'})

state_names  = ['CH', 'PR', 'chl', 'N']
state_labels = ['Carbon (CH)', 'Protein (PR)',
                'Chlorophyll (chl)', 'Nutrient (N)']
colors = ['steelblue', 'darkorange', 'green', 'purple']

print(dat.head())
print(f"\nShape: {dat.shape}")

### Normalize and build Tensors

In [ ]:
t_raw = dat['t'].values.astype(np.float32)
t_min, t_max = t_raw.min(), t_raw.max()
t_norm = (t_raw - t_min) / (t_max - t_min)

scalers = {}
data_scaled = {}
for col in state_names:
    vals = dat[col].values.astype(np.float32)
    vmin, vmax = vals.min(), vals.max()
    scalers[col] = (vmin, vmax)
    data_scaled[col] = (vals - vmin) / (vmax - vmin)

# Observation tensors
t_tensor = tf.constant(t_norm[:, None], dtype=tf.float32)
target   = tf.constant(
    np.column_stack([data_scaled[col] for col in state_names]),
    dtype=tf.float32
)

# Collocation points -- must be a tf.Variable so GradientTape can watch it
t_col = tf.Variable(
    np.linspace(0, 1, 500, dtype=np.float32)[:, None],
    dtype=tf.float32, trainable=False
)
dt_scale = float(t_max - t_min)

print(f"Observation points: {len(t_norm)}")
print(f"Collocation points: {t_col.shape[0]}")


### PINN Model

In [ ]:
# Prior means for ODE parameters
default_params = {
    'mu': 0.3, 'KN': 0.002, 'CHsyn': 5.0, 'm_ex': 10.0,
    'R_ex': 13.0, 'tau': 10.0, 'b': 0.05, 'CNpro': 6.6
}

def build_pinn(hidden=[64, 64, 64], activation='gelu'):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(1,)))
    for h in hidden:
        model.add(tf.keras.layers.Dense(h, activation=activation))
    model.add(tf.keras.layers.Dense(4, activation='sigmoid'))  # sigmoid output
    return model
   

### Physics Loss

In [ ]:
def physics_loss():

    return 
    

### Training Function

In [ ]:
def train_pinn(hidden=[64, 64, 64], lr=1e-3, lam=0.01, activation='gelu',
               epochs=50000, patience=2000, print_every=10000,
               params=None):
    tf.random.set_seed(42)

    if params is None:
        params = default_params

    # Fresh variables each call
    mu    = tf.Variable(params['mu'],    dtype=tf.float32)
    KN    = tf.Variable(params['KN'],    dtype=tf.float32)
    CHsyn = tf.Variable(params['CHsyn'], dtype=tf.float32)
    m_ex  = tf.Variable(params['m_ex'],  dtype=tf.float32)
    R_ex  = tf.Variable(params['R_ex'],  dtype=tf.float32)
    tau   = tf.Variable(params['tau'],   dtype=tf.float32)
    b     = tf.Variable(params['b'],     dtype=tf.float32)
    CNpro = tf.Variable(params['CNpro'], dtype=tf.float32)
    ode_params = [mu, KN, CHsyn, m_ex, R_ex, tau, b, CNpro]

    model = build_pinn(hidden=hidden, activation=activation)
    _ = model(t_tensor)

    all_vars = model.trainable_variables + ode_params
    opt = tf.keras.optimizers.Adam(learning_rate=lr, epsilon=1e-8)

    best_loss, best_weights, wait = np.inf, None, 0
    loss_history = {'total': [], 'data': [], 'physics': []}

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            pred = model(t_tensor)
            loss_data = tf.reduce_mean((pred - target)**2)
            loss_phys = physics_loss(model, t_col, scalers, dt_scale,
                                     ode_params)
            loss = loss_data + lam * loss_phys

        grads = tape.gradient(loss, all_vars)
        opt.apply_gradients(zip(grads, all_vars))

        loss_history['total'].append(loss.numpy())
        loss_history['data'].append(loss_data.numpy())
        loss_history['physics'].append(loss_phys.numpy())

        if loss.numpy() < best_loss:
            best_loss = loss.numpy()
            best_weights = [v.numpy().copy() for v in all_vars]
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % print_every == 0:
                    print(f"[{epoch+1}] data={loss_data.numpy():.5f}  "
                          f"phys={loss_phys.numpy():.5f}  total={loss.numpy():.5f}")
                    param_names = ['mu', 'KN', 'CHsyn', 'm_ex',
                                   'R_ex', 'tau', 'b', 'CNpro']
                    vals = '  '.join(f"{n}={v.numpy():.4f}"
                                     for n, v in zip(param_names, ode_params))
                    print(f"  {vals}")

        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}, "
                  f"best loss: {best_loss:.5f}")
            break

    for v, w in zip(all_vars, best_weights):
        v.assign(w)

    return model, loss_history, ode_params
    

### Train Model

In [ ]:
# import itertools

# # Grid
# hidden_options = [[32, 32], [64, 64], [64, 64, 64], [128, 128]]
# lr_options = [1e-3, 1e-4]
# lam_options = [1.0, 0.1, 0.01]
# activation_options = ['tanh', 'gelu']

# results = []
# param_names = ['mu', 'KN', 'CHsyn',
#                'm_ex', 'R_ex', 'tau', 'b', 'CNpro']

# num_epochs = 1000
# patience_val = 100
# for hidden, lr, lam, act in itertools.product(hidden_options, lr_options,
#                                                lam_options, activation_options):
#     label = f"h={hidden} lr={lr} lam={lam} act={act}"
#     print(f"\n{label}")
#     mdl, hist, ode_p = train_pinn(hidden=hidden, lr=lr, lam=lam,
#                                    activation=act, epochs=num_epochs,
#                                    patience=patience_val, print_every=999999)
#     pred_o = mdl(t_tensor).numpy()
#     mse_per_var = {
#         col: np.mean((data_scaled[col] - pred_o[:, i])**2)
#         for i, col in enumerate(state_names)
#     }
#     mse_total = np.mean(list(mse_per_var.values()))
#     best_data = min(hist['data'])
#     best_phys = hist['physics'][hist['data'].index(best_data)]

#     row = {
#         'hidden': str(hidden), 'lr': lr, 'lam': lam, 'activation': act,
#         'mse_total': mse_total, 'best_data': best_data, 'best_phys': best_phys,
#     }
#     row.update({name: v.numpy() for name, v in zip(param_names, ode_p)})
#     results.append(row)
#     print(f"  mse_total={mse_total:.5f}  data={best_data:.5f}  phys={best_phys:.5f}")

# results_df = pd.DataFrame(results).sort_values('mse_total')
# print("\nTop 10 configurations:")
# print(results_df.head(10).to_string(index=False))


In [ ]:
BEST_HIDDEN = # Modify code here
BEST_LR     = # Modify code here
BEST_LAM    = # Modify code here
BEST_ACTIVATION = # Modify code here

num_epochs = # Modify code here
patience_val = # Modify code here
print("Training PINN...")
model, loss_history, ode_params = train_pinn(
    hidden=BEST_HIDDEN, lr=BEST_LR, lam=BEST_LAM, activation=BEST_ACTIVATION,
    epochs=num_epochs, patience=patience_val, print_every=100
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(loss_history['total'],   label='total',   lw=2)
ax.semilogy(loss_history['data'],    label='data',    lw=2)
ax.semilogy(loss_history['physics'], label='physics', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('PINN Training Loss')
ax.legend()
plt.tight_layout()
plt.show()


### Evaluate and plot

In [ ]:
t_fine     = tf.constant(np.linspace(0, 1, 500, dtype=np.float32)[:, None])
t_fine_raw = t_fine.numpy().flatten() * (t_max - t_min) + t_min

pred_obs  = model(t_tensor).numpy()
pred_fine = model(t_fine).numpy()

def unscale(arr, col):
    vmin, vmax = scalers[col]
    return arr * (vmax - vmin) + vmin

fig, axs = plt.subplots(2, 2, figsize=(13, 8))
for idx, (col, label, color) in enumerate(zip(
        state_names, state_labels, colors)):
    ax = axs.flat[idx]
    obs_phys  = dat[col].values
    pred_phys = unscale(pred_fine[:, idx], col)

    ax.scatter(dat['t'], obs_phys, color='black', s=25, zorder=5, label='observed')
    ax.plot(t_fine_raw, pred_phys, lw=2, color=color, label='PINN')

    rmse = np.sqrt(np.mean((obs_phys - unscale(pred_obs[:, idx], col))**2))
    ax.set_ylabel(label)
    ax.set_xlabel('Time')
    ax.set_title(f'{col}  --  RMSE = {rmse:.4f}')
    ax.legend(fontsize=9)

fig.suptitle(f'PINN -- Macromolecular Model  '
             f'(hidden={BEST_HIDDEN}, lr={BEST_LR}, lam={BEST_LAM})',
             fontsize=13)
plt.tight_layout()
plt.show()


### Print Learned Parameters

In [ ]:
param_names = ['mu', 'KN', 'CHsyn', 'm_ex', 'R_ex', 'tau', 'b', 'CNpro']

print("Learned ODE parameters:")
for name, var in zip(param_names, ode_params):
    prior = default_params[name]
    print(f"  {name:6s} = {var.numpy():.5f}  (prior mean: {prior})")


## 12. $\lambda$ sensitivity analysis

The $\lambda$ hyperparameter controls the tradeoff between data fidelity and
physical consistency. Here we train four models with different $\lambda$ values
and compare their trajectories and MSE values.

- **$\lambda \to 0$**: pure data fitting, no physics constraint
- **$\lambda \to \infty$**: physics dominates, data may be ignored
- **Intermediate $\lambda$**: the PINN regime, where both contribute

In [ ]:
lam_values = [1.0, 0.1, 0.01, 0.001]
lam_models = {}
param_names = ['mu', 'KN', 'CHsyn', 'm_ex', 'R_ex', 'tau', 'b', 'CNpro']

for lam in lam_values:
    print(f"\nTraining lambda = {lam}...")
    mdl, _, ode_p = train_pinn(hidden=BEST_HIDDEN, lr=BEST_LR, lam=lam,
                               activation=BEST_ACTIVATION,
                               epochs=num_epochs, patience=patience_val, print_every=999999)
    pred_f = mdl(t_fine).numpy()
    pred_o = mdl(t_tensor).numpy()
    mse_per_var = {
        col: np.mean((data_scaled[col] - pred_o[:, i])**2)
        for i, col in enumerate(state_names)
    }
    lam_models[lam] = {
        'pred_fine': pred_f,
        'mse': mse_per_var,
        'params': {name: v.numpy() for name, v in zip(param_names, ode_p)}
    }
    print(f"  lambda={lam}  " + "  ".join(
        f"{col}={mse_per_var[col]:.4f}" for col in state_names))
    print(f"  " + "  ".join(
        f"{name}={v.numpy():.4f}" for name, v in zip(param_names, ode_p)))


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
lam_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

for idx, (col, label) in enumerate(zip(state_names, state_labels)):
    ax = axs.flat[idx]
    ax.scatter(dat['t'], dat[col], color='black', s=25,
               zorder=5, label='observed')
    for lam, lc in zip(lam_values, lam_colors):
        pred_phys = unscale(lam_models[lam]['pred_fine'][:, idx], col)
        mse_val = lam_models[lam]['mse'][col]
        ax.plot(t_fine_raw, pred_phys, lw=2, color=lc,
                label=f'$\\lambda$={lam}  (MSE={mse_val:.4f})')
    ax.set_ylabel(label)
    ax.set_xlabel('Time')
    ax.legend(fontsize=7)

fig.suptitle('PINN $\\lambda$ Sensitivity -- Macromolecular Model', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
param_df = pd.DataFrame({lam: lam_models[lam]['params'] for lam in lam_values})
print("Learned parameters by lambda:")
print(param_df.to_string(float_format='%.4f'))
